# FYP Phase 1 — NIDS Prototype (Colab)
This notebook implements **Phase 1** of the project: robust data loading, cleaning, Min-Max scaling, correlation-based feature reduction (CBF), Random Forest + Decision Tree baseline training, validation-based threshold selection targeting low false positives, and artifact export.

**Key fixes vs earlier attempts**
- Collect enough **PortScan** rows (chunk loader; not limited to first 30k).
- Split **by time (per file)** before scaling/feature selection to reduce leakage.
- Fit scaler + CBF **on training only** (no leakage).
- Choose threshold on validation to target **FPR ≤ 3%**.
- Save artifacts for the Streamlit prototype.


In [ ]:
# ==============================
# 0) Colab setup: Drive + paths
# ==============================
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = "/content/drive/MyDrive/FYP_Data"   

FILES = {
    "monday":   "Monday-WorkingHours.pcap_ISCX.csv",
    "ddos":     "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "portscan": "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
}

for k, v in FILES.items():
    p = os.path.join(BASE_PATH, v)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")

print("Loading data from:", BASE_PATH)
print("Files OK:", FILES)


In [ ]:
# ==============================
# 1) Imports + reproducibility
# ==============================
import numpy as np
import pandas as pd
import re
import json
import time
import joblib
from dataclasses import dataclass
from typing import Optional, Tuple, List

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Phase-1 dataset caps (lightweight for IPD)
MAX_BENIGN_TOTAL   = 20000
MAX_DDOS_TOTAL     = 12000
MAX_PORTSCAN_TOTAL = 12000

# Minimum per-attack sanity
MIN_ATTACK_SAMPLES = 500

# Feature reduction config
CBF_CORR_THRESHOLD = 0.95

# Validation constraint target (from your PPRS target: low false positives)
TARGET_MAX_FPR = 0.03

print("Config ready.")


In [ ]:
# ===========================================
# 2) Helpers (robust column + label handling)
# ===========================================
ATTACK_KEYWORDS = [
    'ddos', 'dos', 'attack', 'hulk', 'goldeneye',
    'slowloris', 'slowhttptest', 'heartbleed', 'portscan', 'port scan', 'port-scan'
]

ID_COLUMNS_LIKELY = [
    # identifiers / near-identifiers that cause leakage:
    "flow_id", "source_ip", "destination_ip", "timestamp",
    "src_ip", "dst_ip", "src_address", "dst_address",
]

def canonicalise_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]+", "", regex=True)
    )
    return df

def find_label_col(df: pd.DataFrame) -> Optional[str]:
    for c in df.columns:
        if "label" in c:
            return c
    return None

def normalise_label_text(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("-", " ").replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    mapping = {
        "benign": "benign",
        "normal": "benign",
        "ddos": "ddos",
        "ddos attack": "ddos",
        "dos": "ddos",
        "port scan": "portscan",
        "portscan": "portscan",
    }
    return mapping.get(s, s)

def is_attack_label(label: str) -> bool:
    label = normalise_label_text(label)
    return any(k in label for k in ATTACK_KEYWORDS) and ("benign" not in label)

def safe_numeric(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

def drop_identifier_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    present = [c for c in ID_COLUMNS_LIKELY if c in df.columns]
    if present:
        df = df.drop(columns=present, errors="ignore")
    return df

def correlation_drop_cols(X_train: pd.DataFrame, threshold: float = 0.95) -> List[str]:
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return to_drop

def time_split(df: pd.DataFrame, train_ratio=0.8, val_ratio=0.1):
    n = len(df)
    if n == 0:
        return df.copy(), df.copy(), df.copy()
    i1 = int(train_ratio * n)
    i2 = int((train_ratio + val_ratio) * n)
    return df.iloc[:i1].copy(), df.iloc[i1:i2].copy(), df.iloc[i2:].copy()

def fpr_from_cm(cm: np.ndarray) -> float:
    tn, fp, fn, tp = cm.ravel()
    return fp / (fp + tn) if (fp + tn) > 0 else 0.0


In [ ]:
# ==========================================================
# 3) Chunk loader (fixes PortScan=7 by not truncating)
# ==========================================================
@dataclass
class CollectedData:
    df: pd.DataFrame
    label_col: str

def collect_rows_by_label(csv_path: str, want: str, max_rows: int, chunksize: int = 50000) -> CollectedData:
    collected = []
    label_col_seen = None
    total = 0
    t0 = time.time()

    for chunk in pd.read_csv(csv_path, chunksize=chunksize, low_memory=False):
        chunk = canonicalise_cols(chunk)
        label_col = find_label_col(chunk)
        if label_col is None:
            raise ValueError(f"No label column found in {csv_path}")

        label_col_seen = label_col
        lab = chunk[label_col].astype(str).map(normalise_label_text)

        if want == "benign":
            mask = (lab == "benign")
        else:
            # exact match preferred; safe fallback keeps other attack spellings
            mask = (lab == want) | lab.map(is_attack_label)

        keep = chunk.loc[mask].copy()
        if len(keep) > 0:
            keep[label_col] = lab.loc[mask].values
            collected.append(keep)
            total += len(keep)

        if total >= max_rows:
            break

    out = pd.concat(collected, ignore_index=True).head(max_rows) if collected else pd.DataFrame()
    print(f"Collected {len(out):,} rows for '{want}' from {os.path.basename(csv_path)} in {time.time()-t0:.1f}s")
    return CollectedData(df=out, label_col=label_col_seen)

# ---- Load (robust) ----
path_monday   = os.path.join(BASE_PATH, FILES["monday"])
path_ddos     = os.path.join(BASE_PATH, FILES["ddos"])
path_portscan = os.path.join(BASE_PATH, FILES["portscan"])

monday_data   = collect_rows_by_label(path_monday,   "benign",   MAX_BENIGN_TOTAL)
ddos_data     = collect_rows_by_label(path_ddos,     "ddos",     MAX_DDOS_TOTAL)
portscan_data = collect_rows_by_label(path_portscan, "portscan", MAX_PORTSCAN_TOTAL)

df_monday   = monday_data.df
df_ddos     = ddos_data.df
df_portscan = portscan_data.df

print("\nLabel counts (after normalisation):")
print("Monday:",   df_monday[monday_data.label_col].value_counts().head(5))
print("DDOS:",     df_ddos[ddos_data.label_col].value_counts().head(5))
print("PortScan:", df_portscan[portscan_data.label_col].value_counts().head(5))

if len(df_portscan) < MIN_ATTACK_SAMPLES:
    print(f"WARNING: PortScan samples too low ({len(df_portscan)}). "
          "Phase-1 anomaly model will be primarily DDOS vs benign, but the pipeline supports PortScan when you collect enough rows.")


In [ ]:
# ==================================
# 4) Build Phase-1 datasets + splits
# ==================================
def add_labels(df: pd.DataFrame, attack_type: str, label_col: str) -> pd.DataFrame:
    df = df.copy()
    df["attack_type"] = attack_type
    df["y"] = 0 if attack_type == "benign" else 1
    df["label"] = df[label_col].astype(str).map(normalise_label_text)
    return df

df_monday   = add_labels(df_monday,   "benign",   monday_data.label_col)
df_ddos     = add_labels(df_ddos,     "ddos",     ddos_data.label_col)
df_portscan = add_labels(df_portscan, "portscan", portscan_data.label_col)

m_train, m_val, m_test = time_split(df_monday)
d_train, d_val, d_test = time_split(df_ddos)
p_train, p_val, p_test = time_split(df_portscan)

train_df = pd.concat([m_train, d_train, p_train], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)
val_df   = pd.concat([m_val,   d_val,   p_val],   ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)
test_df  = pd.concat([m_test,  d_test,  p_test],  ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

print("Split sizes:", train_df.shape, val_df.shape, test_df.shape)
print("Train labels:\n", train_df["label"].value_counts())


In [ ]:
# ===========================================
# 5) Cleaning + numeric feature preparation
# ===========================================
DROP_ALWAYS = ["label", "y", "attack_type"]

def prep_features(df: pd.DataFrame):
    df = canonicalise_cols(df)
    df = drop_identifier_columns(df)

    y = df["y"].astype(int)
    attack_type = df["attack_type"].astype(str)

    label_col_any = find_label_col(df)
    drop_cols = set(DROP_ALWAYS)
    if label_col_any is not None:
        drop_cols.add(label_col_any)

    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
    X = safe_numeric(X)

    # drop columns with too many missing values
    too_missing = [c for c in X.columns if X[c].isna().mean() > 0.6]
    if too_missing:
        X = X.drop(columns=too_missing)

    # impute remaining NaNs with median
    X = X.fillna(X.median(numeric_only=True))

    # keep only numeric
    X = X.select_dtypes(include=[np.number]).copy()
    return X, y, attack_type

X_train_raw, y_train, at_train = prep_features(train_df)
X_val_raw,   y_val,   at_val   = prep_features(val_df)
X_test_raw,  y_test,  at_test  = prep_features(test_df)

print("Numeric features (train):", X_train_raw.shape[1])
print("Class balance (train):", np.bincount(y_train))


In [ ]:
# ===================================================
# 6) Correlation-based feature reduction (train only)
# ===================================================
drop_corr = correlation_drop_cols(X_train_raw, threshold=CBF_CORR_THRESHOLD)
print(f"CBF: dropping {len(drop_corr)} features with abs(corr) > {CBF_CORR_THRESHOLD}")

X_train_sel = X_train_raw.drop(columns=drop_corr, errors="ignore")
X_val_sel   = X_val_raw.drop(columns=drop_corr, errors="ignore")
X_test_sel  = X_test_raw.drop(columns=drop_corr, errors="ignore")

common_cols = sorted(set(X_train_sel.columns) & set(X_val_sel.columns) & set(X_test_sel.columns))
X_train_sel = X_train_sel[common_cols]
X_val_sel   = X_val_sel[common_cols]
X_test_sel  = X_test_sel[common_cols]

print("After CBF numeric features:", len(common_cols))


In [ ]:
# ===========================================
# 7) Min-Max scaling (fit on train only)
# ===========================================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_sel)
X_val   = scaler.transform(X_val_sel)
X_test  = scaler.transform(X_test_sel)

print("Scaled shapes:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:
# ===========================================
# 8) Train Decision Tree baseline + Random Forest
# ===========================================
dt = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=5,
    random_state=RANDOM_SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight="balanced"
)

t0 = time.time()
dt.fit(X_train, y_train)
t1 = time.time()
rf.fit(X_train, y_train)
t2 = time.time()

print(f"DT trained in {t1-t0:.2f}s | RF trained in {t2-t1:.2f}s")


In [ ]:
# ===========================================
# 9) Threshold selection (validation, FPR <= 3%)
# ===========================================
rf_val_proba = rf.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 91)

best = None
for th in thresholds:
    y_pred = (rf_val_proba >= th).astype(int)
    cm = confusion_matrix(y_val, y_pred, labels=[0,1])
    fpr = fpr_from_cm(cm)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average="binary", zero_division=0)

    if fpr <= TARGET_MAX_FPR:
        cand = (f1, -fpr, th, prec, rec)
        if best is None or cand > best:
            best = cand

if best is None:
    print("WARNING: no threshold achieved the target FPR on validation. Falling back to best F1.")
    for th in thresholds:
        y_pred = (rf_val_proba >= th).astype(int)
        cm = confusion_matrix(y_val, y_pred, labels=[0,1])
        fpr = fpr_from_cm(cm)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average="binary", zero_division=0)
        cand = (f1, -fpr, th, prec, rec)
        if best is None or cand > best:
            best = cand

best_f1, neg_fpr, best_th, best_prec, best_rec = best
print(f"Chosen threshold: {best_th:.3f} | val F1={best_f1:.3f} | val Precision={best_prec:.3f} | val Recall={best_rec:.3f} | target FPR<= {TARGET_MAX_FPR}")


In [ ]:
# ===========================================
# 10) Evaluation + confusion matrices (ML module style)
# ===========================================
def eval_model(name: str, y_true: np.ndarray, y_pred: np.ndarray):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    fpr = fpr_from_cm(cm)
    print(f"\n=== {name} report ===")
    print(classification_report(y_true, y_pred, digits=3))
    print(f"{name} confusion matrix:\n{cm}")
    print(f"{name} false positive rate (FPR): {fpr:.4f}")
    return cm, fpr

dt_test_pred = dt.predict(X_test)
eval_model("DecisionTree TEST", y_test, dt_test_pred)

rf_test_proba = rf.predict_proba(X_test)[:, 1]
rf_test_pred  = (rf_test_proba >= best_th).astype(int)
eval_model("RandomForest TEST (thresholded)", y_test, rf_test_pred)

disp = ConfusionMatrixDisplay(confusion_matrix(y_test, rf_test_pred, labels=[0,1]), display_labels=["benign","attack"])
disp.plot()


In [ ]:
# =====================================================
# 11) Per-attack-type sanity check (DDOS vs PortScan)
# =====================================================
test_types = pd.Series(at_test).reset_index(drop=True)
y_test_s   = pd.Series(y_test).reset_index(drop=True)
pred_s     = pd.Series(rf_test_pred).reset_index(drop=True)

def type_report(t: str):
    mask = (test_types == t) | (test_types == "benign")
    if mask.sum() == 0:
        print(f"\nNo rows for type '{t}' in test split.")
        return
    y_true = y_test_s[mask].values
    y_pred = pred_s[mask].values
    print(f"\n--- TEST subset: benign vs {t} ---")
    eval_model(f"RF TEST ({t})", y_true, y_pred)

type_report("ddos")
type_report("portscan")


In [ ]:
# ===========================================
# 12) Save artifacts for Streamlit prototype
# ===========================================
ARTIFACT_DIR = "/content/drive/MyDrive/FYP_Artifacts_Phase1"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

meta = {
    "random_seed": RANDOM_SEED,
    "files": FILES,
    "counts": {
        "benign_total": int(len(df_monday)),
        "ddos_total": int(len(df_ddos)),
        "portscan_total": int(len(df_portscan)),
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
    },
    "cbf_corr_threshold": CBF_CORR_THRESHOLD,
    "dropped_corr_features": drop_corr,
    "final_features": common_cols,
    "threshold": float(best_th),
    "target_max_fpr": TARGET_MAX_FPR,
    "timestamp_utc": pd.Timestamp.utcnow().isoformat()
}

joblib.dump(rf, os.path.join(ARTIFACT_DIR, "rf_phase1_model.pkl"))
joblib.dump(dt, os.path.join(ARTIFACT_DIR, "dt_phase1_model.pkl"))
joblib.dump(scaler, os.path.join(ARTIFACT_DIR, "scaler_phase1.pkl"))
with open(os.path.join(ARTIFACT_DIR, "features_phase1.json"), "w") as f:
    json.dump(common_cols, f, indent=2)
with open(os.path.join(ARTIFACT_DIR, "meta_phase1.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Saved artifacts to:", ARTIFACT_DIR)
print("Files:", os.listdir(ARTIFACT_DIR))


In [ ]:
# ===========================================
# 13) Export demo traffic (anonymised)
# ===========================================
demo = test_df.copy()
demo = canonicalise_cols(demo)
demo = drop_identifier_columns(demo)
demo = demo.sample(n=min(2000, len(demo)), random_state=RANDOM_SEED).reset_index(drop=True)

demo_path = os.path.join(ARTIFACT_DIR, "demo_traffic_anonymised.csv")
demo.to_csv(demo_path, index=False)
print("Saved demo traffic:", demo_path, "shape:", demo.shape)
